In [6]:
import os
import re

In [3]:
with open(r'D:\GptFromScratch\data\the_verdict.txt', 'r') as file:
    raw_text = file.read()

In [5]:
print(f"Length of dataset in characters: {len(raw_text):,}")
print(raw_text[:99])

Length of dataset in characters: 20,479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [7]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [16]:
all_words = sorted(set(preprocessed))
all_words.extend(["<|endoftext|>", "<|unk|>"])

In [17]:
vocab_size = len(all_words)
print(f"Vocb Size: {vocab_size}")

Vocb Size: 1132


In [18]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [19]:
for i,item in enumerate(vocab.items()):
    print(item)
    
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [20]:
print(len(vocab.items()))

1132


In [22]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {id: token for token, id in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[token] if token in self.str_to_int else self.str_to_int["<|unk|>"] for token in preprocessed ]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
        

In [23]:
tokenizer = SimpleTokenizer(vocab)
text = """"It's the last he painted, you know,"
 Mrs. Gisburn said with pardonable pride."""
 
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [24]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [25]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [26]:
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [27]:
print(tokenizer.decode(tokenizer.encode(text)))


<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [29]:
import re
import json
from collections import Counter, deque


class BPETokenizer:
    def __init__(self):
        self.vocab = {} # Map token ID to token string. {1126:"some"}
        self.inverse_vocab = {} # Map token string to token ID. {"some":1126}
        self.bpe_merges = {}
        
    def train(self, text, vocab_size, allowed_special=["<|endoftext|>"]):
        
        # Pre-tokenize the text into a list of tokens, including line breaks
        tokens = self.pre_tokenize_text(text)
        
        # Initialize vocab with unique characters, including "Ġ" if present
        # Start with 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
                                char for char in sorted([char for token in tokens for char in token]) if char not in unique_chars
                            )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")
            
        self.vocab = {i:char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char:i for i, char in self.vocab.items()}
        
        # Add allowed special tokens to the vocab
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id
                    
        # Tokenize each pre-token into character ids
        token_id_sequences = [[self.inverse_vocab[char] for char in token] for token in tokens]
        
        # BPE: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_frequent_pairs(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pairs(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id
            
        # Build vocab with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id
            
        
    def encode(self, text):
        tokens = self.pre_tokenize_text(text)
        
        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[token])
            else:
                token_ids.extend(self.tokenize_with_bpe(token))
        return token_ids

    def decode(self, token_ids):
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            if tok in ("\n", "\r"):
                out.append(tok)
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    
    def tokenize_with_bpe(self, token):
        
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, id in zip(token, token_ids) if id is None]
            raise ValueError(f"Token contains unknown characters: {missing_chars}")
        
        while len(token_ids) >= 2:
            # Find the pair with the lowest merge rank (learned earliest)
            pairs = set(zip(token_ids, token_ids[1:]))
            best_pair = min(
                (p for p in pairs if p in self.bpe_merges),
                key=lambda p: self.bpe_merges[p],
                default=None
            )
            if best_pair is None:
                break
            
            # Merge all occurrences of best_pair
            new_id = self.bpe_merges[best_pair]
            new_tokens = []
            i = 0
            while i < len(token_ids):
                if i < len(token_ids) - 1 and (token_ids[i], token_ids[i+1]) == best_pair:
                    new_tokens.append(new_id)
                    i += 2
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            token_ids = new_tokens
            
        return token_ids
            
    @staticmethod
    def pre_tokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue
            
            # Normal chunk without line breaks:
            # - If spaces precede a word, prefix the first word with 'Ġ' and
            #   add standalone 'Ġ' for additional spaces
            # - If spaces trail the chunk (e.g., before a newline) add
            #   standalone 'Ġ' tokens (tiktoken produces id 220 for 'Ġ')
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" + word) # one leading space
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # Trailing spaces (no following word): add standalone 'Ġ' tokens
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens
    
    @staticmethod
    def find_frequent_pairs(token_id_sequences, mode="most"):
        pairs = Counter(
                            pair for token_ids in token_id_sequences for pair in zip(token_ids, token_ids[1:])
                        )
        if not pairs:
            return None
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError(f"Invalid mode: {mode}. Must be 'most' or 'least'.")
        
    @staticmethod
    def replace_pairs(token_id_sequences, pair_id, new_id):
        replaced_sequences = []
        
        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []
            
            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    dq.popleft()  # Remove the next element as it's part of the pair
                else:
                    replaced.append(current)
                    
            replaced_sequences.append(replaced)
        return replaced_sequences
            
    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # Save vocabulary
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # Save BPE merges as a list of dictionaries
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # Load BPE merges
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

In [30]:
tokenizer = BPETokenizer()
tokenizer.train(raw_text, vocab_size=1000, allowed_special=["<|endoftext|>"])

In [36]:
tokenizer.decode(tokenizer.encode("Hello, do you like tea? I am Shah!"))

'Hello, do you like tea? I am Shah!'